# Book Recommendation Dataset — Mục 1, 2 và 3
Theo phần III “Suggestions on how to do the exercise” trong Mid-Term.pdf.
Notebook thực hiện Data Inspection, Data Loading and Integration, Data Cleaning.
Chạy từ đầu đến cuối bằng Run All. Đặt thư mục archive cạnh notebook; trên Colab, tải archive lên /content/archive hoặc sửa DATA_DIR.
Dữ liệu gốc được giữ nguyên. Các quy tắc làm sạch dưới đây là lựa chọn của bài làm, có thể điều chỉnh sau EDA.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
DATA_DIR = Path("archive")
assert DATA_DIR.is_dir(), "Cần đặt thư mục archive trong thư mục làm việc hoặc sửa DATA_DIR."


## 1. Data Inspection
Đọc tự động ba bảng CSV. Kiểm tra UTF-8 bằng chế độ strict trên toàn bộ tệp; giữ ISBN dạng chuỗi để bảo toàn số 0 đầu và ký tự X. Ba bảng khác cấu trúc nên dùng merge thay vì concat.

In [2]:
tables = {}
for name in ["Books", "Users", "Ratings"]:
    path = DATA_DIR / f"{name}.csv"
    with path.open(encoding="utf-8", errors="strict") as stream:
        for _ in stream:
            pass
    tables[name] = pd.read_csv(path, encoding="utf-8", dtype={"ISBN": "string"}, low_memory=False)
books, users, ratings = [tables[n] for n in ["Books", "Users", "Ratings"]]
summary = pd.DataFrame([{"Bảng": n, "Số dòng": len(d), "Số cột": d.shape[1], "Dòng trùng": int(d.duplicated().sum())} for n,d in tables.items()])
display(summary)
for name, frame in tables.items():
    print("\n", name)
    display(pd.DataFrame({"Kiểu": frame.dtypes.astype(str), "Thiếu": frame.isna().sum()}))


,Bảng,Số dòng,Số cột,Dòng trùng
0,Books,271360,8,0
1,Users,278858,3,0
2,Ratings,1149780,3,0



 Books


,Kiểu,Thiếu
ISBN,string,0
Book-Title,str,0
Book-Author,str,2
Year-Of-Publication,str,0
Publisher,str,2
Image-URL-S,str,0
Image-URL-M,str,0
Image-URL-L,str,3



 Users


,Kiểu,Thiếu
User-ID,int64,0
Location,str,0
Age,float64,110762



 Ratings


,Kiểu,Thiếu
User-ID,int64,0
ISBN,string,0
Book-Rating,int64,0


In [3]:
assert books["ISBN"].notna().all() and books["ISBN"].is_unique
assert users["User-ID"].notna().all() and users["User-ID"].is_unique
print("Trùng cặp User-ID, ISBN:", ratings.duplicated(["User-ID", "ISBN"]).sum())
print("Đánh giá thiếu sách:", (~ratings["ISBN"].isin(books["ISBN"])).sum())
print("Đánh giá thiếu người dùng:", (~ratings["User-ID"].isin(users["User-ID"])).sum())
display(ratings["Book-Rating"].value_counts().sort_index().rename("Số lượt").to_frame())
year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
display(books.loc[year.isna(), ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]])
print("Tuổi nhỏ nhất/lớn nhất:", users["Age"].min(), users["Age"].max())


Trùng cặp User-ID, ISBN: 0
Đánh giá thiếu sách: 118644
Đánh giá thiếu người dùng: 0


,Số lượt
Book-Rating,
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
209538,078946697X,"DK Readers: Creating the X-Men, How It All Beg...",2000,DK Publishing Inc,http://images.amazon.com/images/P/078946697X.0...
220731,2070426769,"Peuple du ciel, suivi de 'Les Bergers\"";Jean-M...",2003,Gallimard,http://images.amazon.com/images/P/2070426769.0...
221678,0789466953,"DK Readers: Creating the X-Men, How Comic Book...",2000,DK Publishing Inc,http://images.amazon.com/images/P/0789466953.0...


Tuổi nhỏ nhất/lớn nhất: 0.0 244.0


## 2. Data Loading and Integration
Ratings là bảng trung tâm. Ratings.User-ID → Users.User-ID và Ratings.ISBN → Books.ISBN là hai quan hệ nhiều–một. Left join giữ toàn bộ lượt đánh giá để kiểm tra dữ liệu không khớp; validate='many_to_one' ngăn việc nhân bản dòng do khóa bảng phụ bị trùng. Chỉ ghép các cột cần phân tích để giảm bộ nhớ.

In [4]:
book_columns = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]
integrated_raw = ratings.merge(books[book_columns], on="ISBN", how="left", validate="many_to_one", indicator="book_match")
integrated_raw = integrated_raw.merge(users, on="User-ID", how="left", validate="many_to_one", indicator="user_match")
assert len(integrated_raw) == len(ratings)
print("Kích thước bảng ghép:", integrated_raw.shape)
print(integrated_raw["book_match"].value_counts().to_string())
print(integrated_raw["user_match"].value_counts().to_string())


Kích thước bảng ghép: (1149780, 11)
book_match
both          1031136
left_only      118644
right_only          0
user_match
both          1149780
left_only           0
right_only          0


## 3. Data Cleaning
- Bỏ dòng trùng hoàn toàn; nếu còn nhiều điểm khác nhau cho cùng cặp người dùng–sách thì dừng để xem xét, không tự chọn một điểm.
- Loại riêng 3 dòng sách có năm không phải số khỏi bảng phân tích vì có dấu hiệu lệch trường; không suy đoán để sửa. Các đánh giá liên quan vẫn ở bảng ghép, được đánh dấu không có metadata sách sạch.
- Quy ước tuổi hợp lệ 5–100 và năm xuất bản 1450–2026; đây là ngưỡng đề xuất, không phải kết luận mọi giá trị ngoài khoảng đều sai. Năm 0 coi là chưa biết. Giữ giá trị thiếu dạng số, tạo cờ thiếu; chưa điền median trước khi chia train/test để tránh rò rỉ dữ liệu.
- Điền Unknown cho tác giả, nhà xuất bản và vị trí thiếu/rỗng; chuẩn hóa khoảng trắng. ISBN luôn là chuỗi.
- Giữ điểm 0 ở bảng tổng hợp. Với tập phân loại đề xuất, chỉ lấy điểm 1–10 và đặt liked = 1 nếu điểm ≥ 8. Ý nghĩa của điểm 0 cần xác nhận theo mô tả dataset; không tự gán 0 thành nhãn không thích.


In [5]:
books_clean = books.drop_duplicates().copy()
users_clean = users.drop_duplicates().copy()
ratings_clean = ratings.drop_duplicates().copy()
assert books_clean["ISBN"].is_unique
assert not ratings_clean.duplicated(["User-ID", "ISBN"]).any(), "Cần xem xét đánh giá xung đột."
parsed_year = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
books_quarantine = books_clean.loc[parsed_year.isna()].copy()
books_clean = books_clean.loc[parsed_year.notna()].copy()
y = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
books_clean["Year-Of-Publication"] = y.where(y.between(1450, 2026)).astype("Int64")
books_clean["Year_missing"] = books_clean["Year-Of-Publication"].isna()
a = pd.to_numeric(users_clean["Age"], errors="coerce")
users_clean["Age"] = a.where(a.between(5, 100)).astype("Float64")
users_clean["Age_missing"] = users_clean["Age"].isna()
for frame, cols in [(books_clean, ["Book-Title", "Book-Author", "Publisher"]), (users_clean, ["Location"])]:
    for col in cols:
        frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA).fillna("Unknown")
for frame in [users_clean, ratings_clean]:
    frame["User-ID"] = pd.to_numeric(frame["User-ID"], errors="raise").astype("int64")
ratings_clean["Book-Rating"] = pd.to_numeric(ratings_clean["Book-Rating"], errors="raise")
assert ratings_clean["Book-Rating"].between(0, 10).all()
assert (ratings_clean["Book-Rating"] % 1 == 0).all()
ratings_clean["Book-Rating"] = ratings_clean["Book-Rating"].astype("int8")
print("Sách tách riêng:", len(books_quarantine))
print("Tuổi thiếu sau xử lý:", users_clean["Age"].isna().sum())
print("Năm thiếu sau xử lý:", books_clean["Year-Of-Publication"].isna().sum())


Sách tách riêng: 3
Tuổi thiếu sau xử lý: 112010
Năm thiếu sau xử lý: 4631


In [6]:
data_clean = ratings_clean.merge(books_clean[book_columns + ["Year_missing"]], on="ISBN", how="left", validate="many_to_one", indicator="book_match")
data_clean = data_clean.merge(users_clean, on="User-ID", how="left", validate="many_to_one", indicator="user_match")
assert len(data_clean) == len(ratings_clean)
assert not data_clean.duplicated(["User-ID", "ISBN"]).any()
classification_data = data_clean.loc[data_clean["Book-Rating"].between(1, 10)].copy()
classification_data["liked"] = (classification_data["Book-Rating"] >= 8).astype("int8")
print("Bảng sạch:", data_clean.shape)
print("Lượt đánh giá chưa có metadata sách sạch:", data_clean["book_match"].eq("left_only").sum())
print("Tập phân loại đề xuất:", classification_data.shape)
print(classification_data["liked"].value_counts().sort_index().to_string())
display(data_clean.head())


Bảng sạch: (1149780, 13)
Lượt đánh giá chưa có metadata sách sạch: 118648
Tập phân loại đề xuất: (433671, 14)
liked
0    183784
1    249887


,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Year_missing,book_match,Location,Age,Age_missing,user_match
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,False,both,"tyler, texas, usa",<NA>,True,both
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,False,both,"seattle, washington, usa",<NA>,True,both
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,False,both,"h, new south wales, australia",16.0,False,both
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,False,both,"rijeka, n/a, croatia",16.0,False,both
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,False,both,"rijeka, n/a, croatia",16.0,False,both


### Sử dụng ở bước sau
Các biến books_clean, users_clean, ratings_clean, data_clean và classification_data sẵn sàng cho EDA. Không dùng Book-Rating làm đặc trưng khi dự đoán liked vì nhãn được tạo trực tiếp từ điểm. Sau khi chia tập, fit median/imputer và encoder trên tập train; áp dụng lại lên validation/test. Không dùng User-ID hoặc ISBN như số liên tục. Notebook chưa huấn luyện mô hình vì phạm vi yêu cầu chỉ gồm mục 1–3.